Assignment 3: Maximum Flow, Minimum Cut

Given a directed graph whose edges represent the FLOW CAPACITIES between its vertices, what is the maximum flow that can move between a source and its destination? 

In [1]:
#psuedocode for Ford Fulkerson method 

maximum flow in graph G between vertices s, d:
    R <- residual graph of G
    f(max) <- 0 
    
    while there is an augmenting path in R:
        f(path) <- minimum capacity across augmenting path
        f(max) <- f(max) + f(path)
        saturate augmenting path
        
    return(f(max))
        

SyntaxError: invalid syntax (1389376196.py, line 3)

The flow through an augmenting path is limited by its weakest edge.

When we represent the edges as an array/list, we can use linear search to find the minimum value. 

Saturating the augmenting path: 
    - we place f(path) along the edges, subtracting this value from the flow at each edge (this means one edge ends up at 0)

    - from here, we can find any other augmenting paths that do not hit a zero and saturate them 

In [22]:
#function to find augmenting path 

def find_path(G: list[list[int]], source: int, destination: int) -> list[int] | None:
    """
    Returns any path from source to destination in a directed graph
    represented as an adjacency matrix where 0 means no edge.
    If no path exists, returns None.
    """
    
    # Number of vertices in the graph
    n: int = len(G)
    
    # visited[v] is True once vertex v has been discovered
    visited: list[bool] = [False for _ in range(n)]
    
    # parent[v] stores the vertex from which v was first reached
    # This will allow us to reconstruct the path later
    parent: list[int | None] = [None for _ in range(n)]
    
    # Stack used to implement iterative DFS
    stack: list[int] = [source]
    
    # Mark the start vertex as visited immediately
    visited[source] = True
    
    # Flag to indicate whether we have found the goal
    found: bool = False
    
    # Continue searching while there are vertices to explore
    # and the goal has not yet been found
    while len(stack) > 0 and not found:
        
        # Take the most recently added vertex (DFS behavior)
        u: int = stack.pop()
        
        # If we reached the goal, stop exploring
        if u == destination:
            found = True
        else:
            # Examine all possible outgoing edges u -> v
            v: int = 0
            while v < n:
                
                # If there is an edge from u to v
                if G[u][v] != 0:
                    
                    # If v has not yet been visited, discover it
                    if not visited[v]:
                        visited[v] = True
                        parent[v] = u  # record how we reached v
                        stack.append(v)  # explore v later
                        
                v += 1
                
    # Prepare the result (None unless we found a path)
    path: list[int] | None = None
    
    # If goal was found, reconstruct the path
    if found:
        path: list[int] = []
        current: int | None = destination
        
        # Follow parent links backward from goal to start
        while current is not None:
            path.append(current)
            current = parent[current]
            
        # Reverse to obtain start -> goal order
        path.reverse()
        path = path
    
    return path

In [23]:
#now we use method find_path to find maximum flow across the graph

from copy import deepcopy  # for deep copy of input graph 

def max_flow(G:list[list[int]], source:int, destination:int):
    # Shortcut to size of graph
    n = len(G)
    
    # Initialize return value
    f_max = 0
    
    # Create residual graph. Need deep copy to avoid
    # mutating the input graph G
    R = deepcopy(G)
    
    # Find an initial augmenting path in R
    augmenting = find_path(R, source, destination)
    
    # While there is an augmenting path
    while augmenting is not None:
        
        # f_path = smallest edge of the augmenting path
         
        #use the coordinates of the path to determine what the flows are
        flows = list()
        for i in range(len(augmenting)-1):
            flows.append(R[augmenting[i]][augmenting[i + 1]])
        
        f_path = min(flows) #now find the minimum flow along that path
        
        # add the path's capacity to the graph's max flow
        f_max = f_path + f_max
        
        # Reduce capacity of forward edges in augmenting graph by f_max
        for i in range(len(augmenting) - 1): #for each edge in the augmenting graph 
            R[augmenting[i]][augmenting[i + 1]] = R[augmenting[i]][augmenting[i + 1]] - f_path #reduce capacity by f_path
            
        # Add reverse edges with f_max on the path
        for i in range(len(augmenting) - 1): #for each edge on the augmenting graph 
            R[augmenting[i + 1]][augmenting[i]] =  R[augmenting[i + 1]][augmenting[i]] + f_path #add f_path to track the flow
        
        # find the next augmenting path in R
        augmenting = find_path(R, source, destination)
        
    # Done
    return f_max, R

In [24]:
def reach(graph:list[list[int]], s:int) -> list[int]:
    
    # Shortcuts
    n = len(graph)
    no_edge = graph[0][0]
    reachable = []
    explore_next = [s]
    
    while explore_next:
        u = explore_next.pop()
        if u not in reachable:
            reachable.append(u)
            for v in range(n):
                if graph[u][v] != no_edge:
                    if v not in explore_next:
                        explore_next.append(v)
                        
    return reachable, n


def min_cut(G, R, s):
    
    #start a list for the minimum cut edges
    minimum_cut_edge = []

    #find the reachable vertices and the length of the graph bc I need it for later
    S,n = reach(R, s)
    
    #initalize D
    D = [False] * n
    
    #initialize reachable (S)
    reachable = [False] * n
    
    #compare to S to find the reachable and non-reachable vertices (S and D)
    for i in range(n):
        reachable[i] = (i in S)
        D[i] = (i not in S)
    
    #now we actually find the edges that connect between the reachable and non-reachable sets
    
    for i in range(n): #for each starting vertex 
        
        if reachable[i] == True: #if that vertex is in our "reachable" list
            
            one_row = G[i] #we take the row for that vertex out of our graph 
            
            for j in range(len(one_row)): #look at each entry in the row (potential edge)
                
                if one_row[j] != 0: #if there's an edge there
                    
                    if D[j] == True: #and the connecting vertex is not reachable 
                        
                        minimum_cut_edge.append([i, j]) #add this edge to our minimum cut edges 
    
    #return the list of minimum cut edges
    return(minimum_cut_edge)

    

In [31]:
#testing find_path
graph = [
    # 0   1   2   3   4
    [ 0, 10,  0,  0,  4], # 0
    [ 0,  0,  2,  0,  0], # 1
    [ 0,  0,  0, 10,  0], # 2
    [ 0,  0,  0,  0,  0], # 3
    [ 0,  0,  2,  0,  0]  # 4
]

source_vertex = 0 #give starting vertex
destination_vertex = 3 #give ending vertex
print(f'Path: {find_path(graph, source_vertex, destination_vertex)}') #print resulting path 


#testing max_flow 
f_max, R = max_flow(graph, source_vertex, destination_vertex) #find max flow and R using max_flow function
print(f'Max Flow: {f_max}')


#testing min_cut
print(f'Edges of Minimum Cut: {min_cut(graph, R, source_vertex)}') #find minimum cut using original graph, R, and source vertex 


Path: [0, 4, 2, 3]
Max Flow: 4
Edges of Minimum Cut: [[1, 2], [4, 2]]
